<a href="https://colab.research.google.com/github/Makokung141/Wind-Data/blob/main/GWP5_M2_Table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

เพิ่มข้อมูลรายเดือน

In [ ]:
# 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น
# !pip install --upgrade gspread
from google.colab import auth
import gspread
from google.auth import default
import re

# 2. ยืนยันตัวตน
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 3. เปิด Spreadsheet และเลือกชีตที่ต้องการ
SPREADSHEET_ID = '1vicbYhMVNJN_oRCF2DN-9LhJwufKSMkh-LzVrl7qqGc'
doc = gc.open_by_key(SPREADSHEET_ID)

ws_table = doc.worksheet('Table')
ws_setup = doc.worksheet('Setup')
sheet_id = ws_table.id

# --- ฟังก์ชันแปลงรูปแบบวันที่ให้เป็น "Jul-26", "Aug-26" เสมอ ---
def standardize_month_format(date_str):
    date_str = date_str.strip()
    month_dict = {
        '01': 'Jan', '1': 'Jan', '02': 'Feb', '2': 'Feb', '03': 'Mar', '3': 'Mar',
        '04': 'Apr', '4': 'Apr', '05': 'May', '5': 'May', '06': 'Jun', '6': 'Jun',
        '07': 'Jul', '7': 'Jul', '08': 'Aug', '8': 'Aug', '09': 'Sep', '9': 'Sep',
        '10': 'Oct', '11': 'Nov', '12': 'Dec'
    }

    match_slash = re.match(r'^(\d{1,2})/(\d{2})$', date_str)
    if match_slash:
        m_num, y_num = match_slash.groups()
        if m_num in month_dict:
            return f"{month_dict[m_num]}-{y_num}"

    match_dash = re.match(r'^([A-Za-z]{3})-(\d{2})$', date_str)
    if match_dash:
        m_text, y_num = match_dash.groups()
        return f"{m_text.capitalize()}-{y_num}"

    return date_str
# -----------------------------------------------

# 4. ดึงข้อมูลคอลัมน์ A จากทั้งสองชีต
table_col_a = ws_table.col_values(1)
setup_col_a = ws_setup.col_values(1)

# หาตำแหน่งของ 'Average' และดึงข้อมูลเดือนในหน้า Table
try:
    average_index_0_based = table_col_a.index('Average')
    insert_row_index = average_index_0_based + 1
    table_data_months = [standardize_month_format(val) for val in table_col_a[1:average_index_0_based]]
except ValueError:
    insert_row_index = len(table_col_a) + 1
    table_data_months = [standardize_month_format(val) for val in table_col_a[1:]]

# ข้อมูลเดือนในหน้า Setup
setup_data_months = [standardize_month_format(val) for val in setup_col_a[1:] if val.strip() != '']

# 5. MAPPING: อิงจากเดือนล่าสุดใน Table
new_months_to_add = []

if len(table_data_months) > 0:
    last_table_month = table_data_months[-1]
    print(f"เดือนล่าสุดในหน้า Table คือ: {last_table_month}")

    if last_table_month in setup_data_months:
        setup_last_index = setup_data_months.index(last_table_month)
        new_months_to_add = setup_data_months[setup_last_index + 1:]
    else:
        new_months_to_add = [month for month in setup_data_months if month not in table_data_months]
else:
    new_months_to_add = setup_data_months

# 6. เช็คเงื่อนไขและทำการอัปเดต
if len(new_months_to_add) > 0:
    print(f"เจอเดือนใหม่ที่ต้องอัปเดต จำนวน {len(new_months_to_add)} รายการ: {new_months_to_add}")

    for new_month in new_months_to_add:
        print(f"\nกำลังเตรียมเพิ่มข้อมูลเดือน: {new_month}")

        # แทรกแถวว่าง 1 แถวตรงตำแหน่งก่อน Average
        ws_table.insert_row([], index=insert_row_index)

        row_above_index = insert_row_index - 1

        body = {
            "requests": [
                {
                    # คำสั่งที่ 1: คัดลอกเฉพาะฟอร์แมตและโครงสร้าง A-S (ใช้ PASTE_FORMAT)
                    "copyPaste": {
                        "source": {
                            "sheetId": sheet_id,
                            "startRowIndex": row_above_index - 1,
                            "endRowIndex": row_above_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 18
                        },
                        "destination": {
                            "sheetId": sheet_id,
                            "startRowIndex": insert_row_index - 1,
                            "endRowIndex": insert_row_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 18
                        },
                        "pasteType": "PASTE_FORMAT",
                        "pasteOrientation": "NORMAL"
                    }
                },
                {
                    # คำสั่งที่ 2: คัดลอกเฉพาะรูปแบบ (Format) คอลัมน์ T
                    "copyPaste": {
                        "source": {
                            "sheetId": sheet_id,
                            "startRowIndex": row_above_index - 1,
                            "endRowIndex": row_above_index,
                            "startColumnIndex": 18,
                            "endColumnIndex": 19
                        },
                        "destination": {
                            "sheetId": sheet_id,
                            "startRowIndex": insert_row_index - 1,
                            "endRowIndex": insert_row_index,
                            "startColumnIndex": 18,
                            "endColumnIndex": 19
                        },
                        "pasteType": "PASTE_FORMAT",
                        "pasteOrientation": "NORMAL"
                    }
                },
                {
                    # คำสั่งที่ 3: เอาขอบล่างของเดือนก่อนหน้าออก เพื่อให้เส้นขอบล่างไปอยู่เฉพาะเดือนท้ายสุดเท่านั้น
                    "updateBorders": {
                        "range": {
                            "sheetId": sheet_id,
                            "startRowIndex": row_above_index - 1,
                            "endRowIndex": row_above_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 19
                        },
                        "bottom": {
                            "style": "NONE"
                        }
                    }
                }
            ]
        }

        # ส่งคำสั่งอัปเดตไปยัง Google Sheets
        doc.batch_update(body)

        # พิมพ์ชื่อเดือนใหม่ลงในคอลัมน์ A ของแถวที่เพิ่มเข้ามา
        ws_table.update_cell(insert_row_index, 1, new_month)
        print(f"✅ เพิ่มเดือน '{new_month}' สำเร็จเรียบร้อย!")

        # ขยับ index ลง 1 บรรทัด สำหรับรอบถัดไป
        insert_row_index += 1

    # --- 7. อัปเดตสูตร Average, Min, Max ด้านล่างสุดให้อัตโนมัติ ---
    # โหลดข้อมูลชีต Table ใหม่อีกครั้งเพื่อเช็คตำแหน่งแถวปัจจุบันที่อัปเดตเสร็จแล้ว
    updated_col_a = ws_table.col_values(1)

    try:
        avg_idx = updated_col_a.index('Average') + 1  # ตำแหน่งแถว Average (1-based)
        min_idx = updated_col_a.index('Min') + 1      # ตำแหน่งแถว Min (1-based)
        max_idx = updated_col_a.index('Max') + 1      # ตำแหน่งแถว Max (1-based)

        # หาช่วงข้อมูลเริ่มต้น (สมมติข้อมูลเริ่มต้นแถวข้อมูลแถวแรกคือแถวที่ 9 ตามภาพตัวอย่างของคุณ)
        # และแถวข้อมูลสิ้นสุดคือแถวก่อนถึง Average (avg_idx - 1)
        start_row = 9
        end_row = avg_idx - 1

        print(f"\nกำลังอัปเดตช่วงสูตรสรุป (Average, Min, Max) ให้ครอบคลุมตั้งแต่แถว {start_row} ถึง {end_row}...")

        # วนลูปอัปเดตทีละคอลัมน์ตั้งแต่ B (2) ถึง R (18)
        for col in range(2, 19):
            col_letter = gspread.utils.rowcol_to_a1(1, col)[:-1] # แปลงเลขคอลัมน์เป็นตัวอักษร เช่น B, C, ... S

            # ใส่สูตรใหม่ที่ครอบคลุมถึงแถวเดือนล่าสุดแบบเป๊ะๆ
            ws_table.update_acell(f"{col_letter}{avg_idx}", f"=AVERAGE({col_letter}{start_row}:{col_letter}{end_row})")
            ws_table.update_acell(f"{col_letter}{min_idx}", f"=MIN({col_letter}{start_row}:{col_letter}{end_row})")
            ws_table.update_acell(f"{col_letter}{max_idx}", f"=MAX({col_letter}{start_row}:{col_letter}{end_row})")

        print("✅ อัปเดตสูตร Average, Min, Max เรียบร้อยแล้ว!")

    except ValueError as e:
        print(f"⚠️ หาแถว Average, Min หรือ Max ไม่เจอ: {e}")

    print("\n🎉 อัปเดตข้อมูลและสรุปผลทั้งหมดเสร็จสมบูรณ์!")

else:
    print("✅ ข้อมูลในหน้า Table เป็นข้อมูลล่าสุดแล้ว (ไม่พบเดือนใหม่ในหน้า Setup ถัดจากเดือนล่าสุด)")

เดือนล่าสุดในหน้า Table คือ: Jul-26
✅ ข้อมูลในหน้า Table เป็นข้อมูลล่าสุดแล้ว (ไม่พบเดือนใหม่ในหน้า Setup ถัดจากเดือนล่าสุด)


MoMM

In [ ]:
# 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น
# !pip install --upgrade gspread
from google.colab import auth
import gspread
from google.auth import default
import re

# 2. ยืนยันตัวตน
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 3. เปิด Spreadsheet และเลือกชีตที่ต้องการ
SPREADSHEET_ID = '1vicbYhMVNJN_oRCF2DN-9LhJwufKSMkh-LzVrl7qqGc'
doc = gc.open_by_key(SPREADSHEET_ID)

# เปลี่ยนมาใช้ชีต MOMM ตามที่คุณต้องการ
ws_table = doc.worksheet('MOMM')
ws_setup = doc.worksheet('Setup')
sheet_id = ws_table.id

# --- ฟังก์ชันแปลงรูปแบบวันที่ให้เป็น "Jul-26", "Aug-26" เสมอ ---
def standardize_month_format(date_str):
    date_str = date_str.strip()
    month_dict = {
        '01': 'Jan', '1': 'Jan', '02': 'Feb', '2': 'Feb', '03': 'Mar', '3': 'Mar',
        '04': 'Apr', '4': 'Apr', '05': 'May', '5': 'May', '06': 'Jun', '6': 'Jun',
        '07': 'Jul', '7': 'Jul', '08': 'Aug', '8': 'Aug', '09': 'Sep', '9': 'Sep',
        '10': 'Oct', '11': 'Nov', '12': 'Dec'
    }

    match_slash = re.match(r'^(\d{1,2})/(\d{2})$', date_str)
    if match_slash:
        m_num, y_num = match_slash.groups()
        if m_num in month_dict:
            return f"{month_dict[m_num]}-{y_num}"

    match_dash = re.match(r'^([A-Za-z]{3})-(\d{2})$', date_str)
    if match_dash:
        m_text, y_num = match_dash.groups()
        return f"{m_text.capitalize()}-{y_num}"

    return date_str
# -----------------------------------------------

# 4. ดึงข้อมูลคอลัมน์ A จากทั้งสองชีต
table_col_a = ws_table.col_values(1)
setup_col_a = ws_setup.col_values(1)

# หาตำแหน่งของ 'Average' และดึงข้อมูลเดือนในหน้า MOMM
try:
    average_index_0_based = table_col_a.index('Average')
    insert_row_index = average_index_0_based + 1
    table_data_months = [standardize_month_format(val) for val in table_col_a[1:average_index_0_based] if val.strip() != '']
except ValueError:
    insert_row_index = len(table_col_a) + 1
    table_data_months = [standardize_month_format(val) for val in table_col_a[1:] if val.strip() != '']

# ข้อมูลเดือนในหน้า Setup
setup_data_months = [standardize_month_format(val) for val in setup_col_a[1:] if val.strip() != '']

# 5. MAPPING: อิงจากเดือนล่าสุดในหน้า MOMM
new_months_to_add = []

if len(table_data_months) > 0:
    last_table_month = table_data_months[-1]
    print(f"เดือนล่าสุดในหน้า MOMM คือ: {last_table_month}")

    if last_table_month in setup_data_months:
        setup_last_index = setup_data_months.index(last_table_month)
        new_months_to_add = setup_data_months[setup_last_index + 1:]
    else:
        new_months_to_add = [month for month in setup_data_months if month not in table_data_months]
else:
    new_months_to_add = setup_data_months

# 6. เช็คเงื่อนไขและทำการอัปเดต
if len(new_months_to_add) > 0:
    print(f"เจอเดือนใหม่ที่ต้องอัปเดต จำนวน {len(new_months_to_add)} รายการ: {new_months_to_add}")

    for new_month in new_months_to_add:
        print(f"\nกำลังเตรียมเพิ่มข้อมูลเดือน: {new_month}")

        # แทรกแถวว่าง 1 แถวตรงตำแหน่งก่อน Average ในหน้า MOMM
        ws_table.insert_row([], index=insert_row_index)

        row_above_index = insert_row_index - 1

        body = {
            "requests": [
                {
                    # คัดลอกเฉพาะฟอร์แมตและโครงสร้าง ตั้งแต่คอลัมน์ A ถึง AC (0 ถึง 29)
                    "copyPaste": {
                        "source": {
                            "sheetId": sheet_id,
                            "startRowIndex": row_above_index - 1,
                            "endRowIndex": row_above_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 27  # คอลัมน์ AC (0-based index ถึง 29)
                        },
                        "destination": {
                            "sheetId": sheet_id,
                            "startRowIndex": insert_row_index - 1,
                            "endRowIndex": insert_row_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 27
                        },
                        "pasteType": "PASTE_FORMAT",
                        "pasteOrientation": "NORMAL"
                    }
                },
                {
                    # จัดการเส้นขอบล่างของแถวก่อนหน้าให้หายไป ให้เหลือขอบล่างเฉพาะเดือนท้ายสุด
                    "updateBorders": {
                        "range": {
                            "sheetId": sheet_id,
                            "startRowIndex": row_above_index - 1,
                            "endRowIndex": row_above_index,
                            "startColumnIndex": 0,
                            "endColumnIndex": 27
                        },
                        "bottom": {
                            "style": "NONE"
                        }
                    }
                }
            ]
        }

        # ส่งคำสั่งอัปเดตไปยัง Google Sheets
        doc.batch_update(body)

        # พิมพ์ชื่อเดือนใหม่ลงในคอลัมน์ A ของแถวที่เพิ่มเข้ามา
        ws_table.update_cell(insert_row_index, 1, new_month)
        print(f"✅ เพิ่มเดือน '{new_month}' ในหน้า MOMM สำเร็จเรียบร้อย!")

        # ขยับ index ลง 1 บรรทัด สำหรับรอบถัดไป
        insert_row_index += 1

    print("\n🎉 อัปเดตข้อมูลและจัดรูปแบบในหน้า MOMM เสร็จสมบูรณ์!")

else:
    print("✅ ข้อมูลในหน้า MOMM เป็นข้อมูลล่าสุดแล้ว (ไม่พบเดือนใหม่ในหน้า Setup ถัดจากเดือนล่าสุด)")

เดือนล่าสุดในหน้า MOMM คือ: Jul-26
✅ ข้อมูลในหน้า MOMM เป็นข้อมูลล่าสุดแล้ว (ไม่พบเดือนใหม่ในหน้า Setup ถัดจากเดือนล่าสุด)


Log

In [ ]:
import pandas as pd
import numpy as np
import gspread
from google.colab import auth
from google.auth import default
import re
import string
import datetime

# 1. ยืนยันตัวตนเข้าถึง Google Sheets
print("Requesting Google Sheets access...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Connection successful!\n")

def col2num(col_str):
    num = 0
    for c in col_str:
        if c in string.ascii_letters:
            num = num * 26 + (ord(c.upper()) - ord('A')) + 1
    return num - 1

def clean_superscript_string(text):
    cleaned = str(text).strip()
    cleaned = re.sub(r'[⁰¹²³⁴⁵⁶⁷⁸⁹]', '', cleaned)
    cleaned = re.sub(r'\[\d+\]', '', cleaned)
    return cleaned.strip()

# ปรับปรุงฟังก์ชันให้รองรับ Nov-22 เปลี่ยนเป็น 11/2022 อย่างครอบคลุม
def convert_month_format(month_str):
    if not month_str:
        return ""
    month_str = month_str.strip()

    # แบบ 1: Nov-22 -> 11/2022
    try:
        date_obj = datetime.datetime.strptime(month_str, '%b-%y')
        return date_obj.strftime('%m/%Y')
    except ValueError:
        pass

    # แบบ 2: Nov-2022 -> 11/2022
    try:
        date_obj = datetime.datetime.strptime(month_str, '%b-%Y')
        return date_obj.strftime('%m/%Y')
    except ValueError:
        pass

    # แบบ 3: 11/22 -> 11/2022
    try:
        date_obj = datetime.datetime.strptime(month_str, '%m/%y')
        return date_obj.strftime('%m/%Y')
    except ValueError:
        pass

    # แบบ 4: 11/2022 (ตรงตัว)
    try:
        datetime.datetime.strptime(month_str, '%m/%Y')
        return month_str
    except ValueError:
        pass

    # ถ้าแปลงไม่ได้เลย คืนค่าเดิมกลับไป
    return month_str

def group_consecutive_dates(date_list):
    if not date_list:
        return ""
    parsed_dates = []
    for d in date_list:
        try:
            dt = pd.to_datetime(d, dayfirst=True).date()
            parsed_dates.append(dt)
        except Exception:
            pass
    if not parsed_dates:
        return ", ".join(date_list)

    parsed_dates = sorted(list(set(parsed_dates)))
    ranges = []
    start_date = parsed_dates[0]
    end_date = parsed_dates[0]

    for i in range(1, len(parsed_dates)):
        if (parsed_dates[i] - end_date).days == 1:
            end_date = parsed_dates[i]
        else:
            if start_date == end_date:
                ranges.append(start_date.strftime('%d/%m/%Y'))
            else:
                ranges.append(f"{start_date.strftime('%d/%m/%Y')} - {end_date.strftime('%d/%m/%Y')}")
            start_date = parsed_dates[i]
            end_date = parsed_dates[i]

    if start_date == end_date:
        ranges.append(start_date.strftime('%d/%m/%Y'))
    else:
        ranges.append(f"{start_date.strftime('%d/%m/%Y')} - {end_date.strftime('%d/%m/%Y')}")
    return ", ".join(ranges)

# === ปรับเปลี่ยนชื่ออุปกรณ์ WS และ WD ===
def get_sensor_name(clean_data_header_row, clean_data_col_original_idx):
    full_col_name = str(clean_data_header_row[clean_data_col_original_idx]).strip()

    match = re.search(r'Ch\d+_(Anem|Vane)_?(\d+\.?\d*)m?_?([A-Za-z]{2,4})?', full_col_name, re.IGNORECASE)

    if match:
        device_type = match.group(1)
        height_raw = match.group(2)
        direction_raw = match.group(3)

        category_prefix = ""
        if "anem" in device_type.lower():
            category_prefix = "WS"
        elif "vane" in device_type.lower():
            category_prefix = "WD"

        height_str = ""
        if height_raw:
            height_str = height_raw.split('.')[0] if '.' in height_raw else height_raw

        direction_str = direction_raw.upper() if direction_raw else ""

        return f"{category_prefix}{height_str}{direction_str}".strip()
    else:
        return full_col_name

target_columns_letters = ['B', 'G', 'L', 'Q', 'V', 'AA', 'AF', 'AK', 'AN', 'AQ', 'BF', 'AX', 'BB']
target_cols_idx = [col2num(c) for c in target_columns_letters]

CLEAN_COL_TO_TABLE_COL = {
    0: 8, 1: 9, 2: 10, 3: 11, 4: 12, 5: 13, 6: 14, 7: 15, 8: 16, 9: 17, 10: 18, 11: 19, 12: 20
}

REMARK_COL_IDX = col2num('T')  # Index 19

# === ฟังก์ชันเช็คว่า DRR เป็น 100 หรือไม่ ===
def is_drr_100(drr_str):
    if not drr_str:
        return False
    cleaned_str = str(drr_str).replace('%', '').replace(',', '').strip()
    try:
        return float(cleaned_str) == 100.0
    except ValueError:
        return False
# ==========================================

def main():
    # 2. URL ของชีท Master
    master_url = "https://docs.google.com/spreadsheets/d/1vicbYhMVNJN_oRCF2DN-9LhJwufKSMkh-LzVrl7qqGc/edit?usp=sharing"
    master_id = re.search(r'/d/([a-zA-Z0-9-_]+)', master_url).group(1)
    master_wb = gc.open_by_key(master_id)

    ws_table = master_wb.worksheet("Table")
    table_data = ws_table.get_all_values()

    # ดึงค่า Commissioning Date จากชีท Setup ช่อง G2
    ws_setup = master_wb.worksheet("Setup")
    COMMISSIONING_DATE_STR = ws_setup.acell('G2').value.strip()
    comm_date_obj = datetime.datetime.strptime(COMMISSIONING_DATE_STR, '%d/%m/%Y').date()

    # ล้างค่าคอลัมน์ T (Remark) เก่าทั้งหมด
    for r in range(len(table_data)):
        while len(table_data[r]) <= REMARK_COL_IDX:
            table_data[r].append("")
        if r >= 8:
            table_data[r][REMARK_COL_IDX] = ""

    col_a_data = [row[0] for row in table_data]

    # ใช้ Commissioning Month จากที่ดึงมาด้านบน
    commissioning_month = comm_date_obj.strftime('%m/%Y')
    print(f"📌 Using Commissioning Date from Setup! G2: {COMMISSIONING_DATE_STR} (Month: {commissioning_month})\n")

    all_setup_data = ws_setup.get_all_values()

    if not all_setup_data:
        df_setup = pd.DataFrame()
    else:
        original_headers = all_setup_data[0]
        data_rows = all_setup_data[1:]
        unique_headers = []
        counts = {}
        for h in original_headers:
            if h in counts:
                counts[h] += 1
                unique_headers.append(f"{h}_{counts[h]}")
            else:
                counts[h] = 0
                unique_headers.append(h)
        df_setup = pd.DataFrame(data_rows, columns=unique_headers)

    log_rows = []
    event_counter = 1

    def get_issue_mask(df):
        is_na = df.isin(['N/A', '#N/A', 'NA', '#N/A!', 'N/A ', ' N/A'])
        is_blank = (df == '') | (df.isna()) | (df.isnull())
        return is_na | is_blank

    print("Starting to process and analyze Clean Data for each month...")

    month_row_mapping = {}
    for i, val in enumerate(col_a_data):
        formatted_month = convert_month_format(val)
        if formatted_month:
            month_row_mapping[formatted_month] = i

    for index, row in df_setup.iterrows():
        raw_month = str(row.get('MM/YYYY', '')).strip()
        month_cleaned = clean_superscript_string(raw_month)

        # === แปลงเดือนที่ได้จาก Setup ให้เป็น format MM/YYYY ก่อนการเช็คเงื่อนไข ===
        month = convert_month_format(month_cleaned)

        url = row.get('URL', '')

        if not url or pd.isna(url) or url == '':
            continue

        # แจ้งชื่อเดือนที่กำลังประมวลผล
        print(f"Processing month: {month} (Original: {month_cleaned}) ...")

        is_comm_date = (month == commissioning_month)
        table_row_idx = month_row_mapping.get(month, -1)
        skip_local_sensors = False
        current_month_issues = []

        # 1. กรณีเดือน Commissioning Date
        if is_comm_date and table_row_idx != -1:
            current_month_issues.append(f"Commissioning Date: {COMMISSIONING_DATE_STR}")

        try:
            sheet_id = re.search(r'/d/([a-zA-Z0-9-_]+)', url).group(1)
            wb = gc.open_by_key(sheet_id)

            # โหลด Clean Data
            ws_clean = wb.worksheet("Clean Data")
            raw_clean_data = ws_clean.get_all_values()

            # โหลด Data ดิบ (เพื่อเปรียบเทียบว่าข้อมูลถูกคลีน หรือ หายไปจริงๆ)
            try:
                ws_raw = wb.worksheet("Data")
                raw_raw_data = ws_raw.get_all_values()
            except gspread.exceptions.WorksheetNotFound:
                print(f"  -> Warning: 'Data' sheet not found in {month}. Treating all missing as 'data missing'.")
                raw_raw_data = []

            if len(raw_clean_data) >= 10:
                df_clean = pd.DataFrame(raw_clean_data)
                valid_cols = [c for c in target_cols_idx if c < len(df_clean.columns)]

                if len(raw_raw_data) >= 10:
                    df_raw = pd.DataFrame(raw_raw_data)
                    min_rows = min(len(df_clean), len(df_raw))
                    df_clean = df_clean.iloc[:min_rows]
                    df_raw = df_raw.iloc[:min_rows]
                else:
                    df_raw = pd.DataFrame(np.nan, index=df_clean.index, columns=df_clean.columns)

                dates_col = df_clean.iloc[5:, 0].astype(str).str.strip().str.split(' ').str[0]

                df_sensors_clean = df_clean.iloc[5:, valid_cols]

                valid_cols_raw = [c for c in valid_cols if c < len(df_raw.columns)]
                df_sensors_raw = pd.DataFrame(index=df_sensors_clean.index, columns=valid_cols, dtype=object)
                for i, vc in enumerate(valid_cols):
                    if vc in valid_cols_raw:
                        df_sensors_raw.iloc[:, i] = df_raw.iloc[5:, vc].values
                    else:
                        df_sensors_raw.iloc[:, i] = np.nan

                valid_dates = [d for d in dates_col.unique() if str(d).strip()]
                total_days_in_month = len(valid_dates)

                # === หา Mask ความผิดปกติเบื้องต้น ===
                clean_issue_mask = get_issue_mask(df_sensors_clean)
                raw_issue_mask = get_issue_mask(df_sensors_raw)

                # === ยกเว้น (Ignore) วันที่ก่อนหน้า Commissioning Date ===
                parsed_dates_series = pd.to_datetime(dates_col, dayfirst=True, errors='coerce').dt.date
                before_comm_mask = (parsed_dates_series < comm_date_obj).fillna(False).astype(bool)

                clean_issue_mask.loc[before_comm_mask, :] = False
                raw_issue_mask.loc[before_comm_mask, :] = False
                # ====================================================

                is_missing_mask = clean_issue_mask & raw_issue_mask
                is_cleaned_mask = clean_issue_mask & ~raw_issue_mask

                # 2. Global Event (Data logger)
                row_all_missing = is_missing_mask.all(axis=1)
                if row_all_missing.any():
                    dates_logger_broken = dates_col[row_all_missing].unique().tolist()
                    dates_logger_broken = [d for d in dates_logger_broken if str(d).strip()]

                    if dates_logger_broken and table_row_idx != -1:
                        dates_str = group_consecutive_dates(dates_logger_broken)
                        skip_local_sensors = (len(dates_logger_broken) == total_days_in_month and total_days_in_month > 0)
                        current_month_issues.append(f"Data Logger ไม่ส่งข้อมูล: {dates_str}")

                # 3. Local Event (สำหรับอุปกรณ์รายตัว)
                if not skip_local_sensors:
                    local_missing_mask = is_missing_mask & ~row_all_missing.values[:, None]
                    local_cleaned_mask = is_cleaned_mask & ~row_all_missing.values[:, None]

                    for col_idx in range(len(valid_cols)):
                        target_table_col = CLEAN_COL_TO_TABLE_COL.get(col_idx, -1)
                        if target_table_col == -1: continue

                        sensor_name = get_sensor_name(raw_clean_data[0], valid_cols[col_idx])

                        if not (sensor_name.startswith("WD") or sensor_name.startswith("WS")):
                            continue

                        if table_row_idx != -1:
                            if target_table_col < len(table_data[table_row_idx]):
                                drr_value_in_table = table_data[table_row_idx][target_table_col]
                                if is_drr_100(drr_value_in_table):
                                    continue

                        # ตรวจสอบ Data Missing
                        col_missing = local_missing_mask.iloc[:, col_idx]
                        if col_missing.any():
                            dates_broken = dates_col[col_missing].unique().tolist()
                            dates_broken = [d for d in dates_broken if str(d).strip()]

                            if dates_broken and table_row_idx != -1:
                                dates_str = group_consecutive_dates(dates_broken)
                                current_month_issues.append(f"{sensor_name} data missing: {dates_str}")

                        # ตรวจสอบ ข้อมูลที่ถูกคลีน (data cleaned)
                        col_cleaned = local_cleaned_mask.iloc[:, col_idx]
                        if col_cleaned.any():
                            dates_cleaned = dates_col[col_cleaned].unique().tolist()
                            dates_cleaned = [d for d in dates_cleaned if str(d).strip()]

                            if dates_cleaned and table_row_idx != -1:
                                dates_str = group_consecutive_dates(dates_cleaned)
                                current_month_issues.append(f"{sensor_name} data cleaned: {dates_str}")

        except Exception as e:
            print(f"  -> Read Error: {e}")

        if current_month_issues and table_row_idx != -1:
            table_data[table_row_idx][REMARK_COL_IDX] = f"[{event_counter}]"
            combined_details = "\n".join([f"- {issue}" for issue in current_month_issues])
            log_rows.append((event_counter, month, combined_details))
            event_counter += 1

    # 4. เขียนอัปเดตกลับไปยังชีท Table
    print("\nUpdating Table sheet...")
    last_data_row = len(table_data)
    ws_table.update(values=table_data, range_name=f'A1:{gspread.utils.rowcol_to_a1(last_data_row, len(table_data[0]))}')

    # 5. จัดการชีท Log
    print("Updating Log sheet...")
    try:
        old_log_ws = master_wb.worksheet("Log")
        master_wb.del_worksheet(old_log_ws)
    except gspread.exceptions.WorksheetNotFound:
        pass

    ws_log = master_wb.add_worksheet(title="Log", rows=max(100, len(log_rows) + 5), cols=12)

    if log_rows:
        log_sheet_payload = [["Remark No.", "MM/YY", "Detail"]]
        for remark_no, log_month, detail in log_rows:
            log_sheet_payload.append([str(remark_no), log_month, detail])

        ws_log.update(values=log_sheet_payload, range_name=f'A1:C{len(log_sheet_payload)}')

        requests = []
        sheet_log_id = ws_log.id

        for r_idx in range(len(log_sheet_payload)):
            requests.append({
                "mergeCells": {
                    "range": {
                        "sheetId": sheet_log_id,
                        "startRowIndex": r_idx,
                        "endRowIndex": r_idx + 1,
                        "startColumnIndex": 2,
                        "endColumnIndex": 12
                    },
                    "mergeType": "MERGE_ALL"
                }
            })

        master_wb.batch_update({"requests": requests})

    print("Process completed successfully! ✅")

if __name__ == '__main__':
    main()

Requesting Google Sheets access...
Connection successful!

📌 Using Commissioning Date from Setup! G2: 13/10/2022 (Month: 10/2022)

Starting to process and analyze Clean Data for each month...
Processing month: 10/2022 (Original: Oct-22) ...
Processing month: 11/2022 (Original: Nov-22) ...
  -> Read Error: APIError: [503]: The service is currently unavailable.
Processing month: 12/2022 (Original: Dec-22) ...


/tmp/ipykernel_872/2507648169.py:73: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(d, dayfirst=True).date()


Processing month: 01/2023 (Original: Jan-23) ...
Processing month: 02/2023 (Original: Feb-23) ...
Processing month: 03/2023 (Original: Mar-23) ...
Processing month: 04/2023 (Original: Apr-23) ...
Processing month: 05/2023 (Original: May-23) ...
Processing month: 06/2023 (Original: Jun-23) ...
Processing month: 07/2023 (Original: Jul-23) ...
Processing month: 08/2023 (Original: Aug-23) ...
Processing month: 09/2023 (Original: Sep-23) ...
Processing month: 10/2023 (Original: Oct-23) ...
Processing month: 11/2023 (Original: Nov-23) ...
Processing month: 12/2023 (Original: Dec-23) ...
Processing month: 01/2024 (Original: Jan-24) ...
Processing month: 02/2024 (Original: Feb-24) ...
Processing month: 03/2024 (Original: Mar-24) ...
Processing month: 04/2024 (Original: Apr-24) ...
Processing month: 05/2024 (Original: May-24) ...
Processing month: 06/2024 (Original: Jun-24) ...
Processing month: 07/2024 (Original: Jul-24) ...
Processing month: 08/2024 (Original: Aug-24) ...
Processing month: 09

/tmp/ipykernel_872/2507648169.py:73: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(d, dayfirst=True).date()


Processing month: 05/2025 (Original: May-25) ...
Processing month: 06/2025 (Original: Jun-25) ...
Processing month: 07/2025 (Original: Jul-25) ...
Processing month: 08/2025 (Original: Aug-25) ...
Processing month: 09/2025 (Original: Sep-25) ...
Processing month: 10/2025 (Original: Oct-25) ...
Processing month: 11/2025 (Original: Nov-25) ...


/tmp/ipykernel_872/2507648169.py:73: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(d, dayfirst=True).date()


Processing month: 12/2025 (Original: Dec-25) ...
Processing month: 01/2026 (Original: Jan-26) ...
Processing month: 02/2026 (Original: Feb-26) ...
Processing month: 03/2026 (Original: Mar-26) ...
Processing month: 04/2026 (Original: Apr-26) ...
Processing month: 05/2026 (Original: May-26) ...
Processing month: 06/2026 (Original: Jun-26) ...
Processing month: 07/2026 (Original: Jul-26) ...

Updating Table sheet...
Updating Log sheet...
Process completed successfully! ✅


WD22.**5**

In [ ]:
# 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น

import gspread

import numpy as np

import re

from google.colab import auth

from google.auth import default



# ==========================================

# ⚠️ ตั้งค่าตรงนี้ก่อนรัน (ตั้งค่า URL และ ช่วงเซลล์)

# ==========================================

# ใส่ URL ของไฟล์หลักของคุณ

MAIN_SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1vicbYhMVNJN_oRCF2DN-9LhJwufKSMkh-LzVrl7qqGc/edit?usp=sharing"

# ระบุช่วงเซลล์ "ต้นทาง" และ "ปลายทาง" สำหรับตารางที่ต้องการดึง

SOURCE_RANGE1 = "I3:Z26"

TARGET_RANGE1 = "C5:T28"



# ตั้งชื่อชีตให้ตรงกับไฟล์หลัก

SETUP_SHEET_NAME = "Setup"

SUMMARY_SHEET_NAME = "Summary_WD22.5"

# ==========================================



# ฟังก์ชันแปลง A1 Notation เพื่อหาขนาด (แถว, คอลัมน์) ที่แน่นอน

def get_dimensions(range_str):

    match = re.match(r'([A-Z]+)(\d+):([A-Z]+)(\d+)', range_str)

    if match:

        c1, r1, c2, r2 = match.groups()

        def col_to_num(col):

            return sum((ord(char) - 64) * (26 ** i) for i, char in enumerate(reversed(col)))

        return (int(r2) - int(r1) + 1, col_to_num(c2) - col_to_num(c1) + 1)

    return 0, 0



# ฟังก์ชันดึงค่าและตรวจสอบว่าใช้คำนวณได้หรือไม่ (แยกค่า Error ออก)

def parse_values(raw_values, expected_rows, expected_cols):

    matrix = np.zeros((expected_rows, expected_cols))

    valid_mask = np.zeros((expected_rows, expected_cols)) # เก็บค่า 1 ถ้าเป็นตัวเลขที่นำมาคำนวณได้



    for r in range(min(expected_rows, len(raw_values))):

        for c in range(min(expected_cols, len(raw_values[r]))):

            val = raw_values[r][c]



            if isinstance(val, (int, float)):

                matrix[r, c] = val

                valid_mask[r, c] = 1 # นับว่าเป็นค่าที่ถูกต้อง



            elif isinstance(val, str):

                val_str = val.strip()

                # ข้ามค่าว่างและ Error ของ Google Sheets (ที่ขึ้นต้นด้วย # เช่น #N/A, #DIV/0!, #REF!)

                if val_str == "" or val_str.startswith('#'):

                    continue



                try:

                    matrix[r, c] = float(val_str.replace(',', ''))

                    valid_mask[r, c] = 1 # นับว่าเป็นค่าที่ถูกต้อง

                except ValueError:

                    pass # หากแปลงเป็นตัวเลขไม่ได้ ให้ข้ามไป



    return matrix, valid_mask



def main():

    print("🔄 กำลังตรวจสอบสิทธิ์การเข้าถึงบัญชี Google...")

    auth.authenticate_user()

    creds, _ = default()

    gc = gspread.authorize(creds)

    print("✅ เข้าสู่ระบบสำเร็จ\n")



    print(f"กำลังเริ่มดึงข้อมูลของ {SETUP_SHEET_NAME}... ระบบกำลังประมวลผล\n")



    try:

        main_ss = gc.open_by_url(MAIN_SPREADSHEET_URL)

    except Exception as e:

        print(f"❌ ข้อผิดพลาด: ไม่สามารถเปิดไฟล์หลักได้ โปรดตรวจสอบ URL\nรายละเอียด: {e}")

        return



    try:

        setup_sheet = main_ss.worksheet(SETUP_SHEET_NAME)

        summary_sheet = main_ss.worksheet(SUMMARY_SHEET_NAME)

    except gspread.exceptions.WorksheetNotFound as e:

        print(f"❌ ข้อผิดพลาด: หาชีตชื่อไม่พบ ({e})\nโปรดตรวจสอบว่าตั้งชื่อแท็บถูกต้องและไม่มีช่องว่างเกินมา")

        return



   # ดึงข้อมูลจาก Setup (ข้ามแถวแรกที่เป็น Header)
    setup_data = setup_sheet.get_all_values()[1:]

    ROWS1, COLS1 = get_dimensions(SOURCE_RANGE1)

    # เก็บผลรวม
    sum_matrix1 = np.zeros((ROWS1, COLS1))

    # เก็บจำนวนครั้งที่พบข้อมูลถูกต้อง (Cell-by-cell counting)
    count_matrix1 = np.zeros((ROWS1, COLS1))

    file_count1 = 0

    for i, row in enumerate(setup_data):
        # คอลัมน์ B=1 (URL), C=2 (Sheet Name), E=4 (Checkbox)
        url1 = row[1].strip() if len(row) > 1 else ""
        sheet_name1 = row[2].strip() if len(row) > 2 else ""

        # ตรวจสอบค่า Checkbox ในคอลัมน์ E ว่าถูกติ๊กหรือไม่ (ดึงค่ามาเป็น text จึงเช็คด้วยคำว่า 'TRUE')
        is_checked = False
        if len(row) > 4:
            is_checked = row[4].strip().upper() == 'TRUE'

        row_num = i + 2 # อ้างอิงตามบรรทัดใน Google Sheets (บวก 2 เพราะข้าม Header และ index เริ่มจาก 0)

        # เงื่อนไข: ทำงานก็ต่อเมื่อมี URL, ชื่อชีต และ Checkbox ถูกติ๊กแล้วเท่านั้น
        if url1 and sheet_name1 and is_checked:
            try:
                source_ss1 = gc.open_by_url(url1)
                source_sheet1 = source_ss1.worksheet(sheet_name1)
                raw_data1 = source_sheet1.get(SOURCE_RANGE1, value_render_option='UNFORMATTED_VALUE')

                matrix1, valid1 = parse_values(raw_data1, ROWS1, COLS1)
                sum_matrix1 += matrix1
                count_matrix1 += valid1 # บวกจำนวนตัวหารเฉพาะช่องที่มีค่า
                file_count1 += 1
            except gspread.exceptions.WorksheetNotFound:
                print(f"⚠️ ไม่พบชีตชื่อ '{sheet_name1}' ในไฟล์บรรทัดที่ {row_num}")
            except Exception as e:
                print(f"⚠️ Error (บรรทัด {row_num}): {e}")

        elif url1 and sheet_name1 and not is_checked:
            # กรณีที่ไม่ได้ติ๊ก Checkbox ให้แสดงข้อความแจ้งเตือนว่าข้ามบรรทัดนี้ไป
            print(f"⏩ ข้ามบรรทัดที่ {row_num}: เนื่องจากไม่ได้ติ๊กเลือกในคอลัมน์ E")



    # ==========================================

    # สรุปและเขียนกลับไปยัง Summary

    # ==========================================

    print("\n" + "="*30)

    print("📊 ผลการดึงข้อมูล:")



    # ฟังก์ชันหาค่าเฉลี่ย โดยพิจารณาตัวหารแบบช่องต่อช่อง

    def format_average(sum_mat, count_mat):

        result = []

        for r in range(sum_mat.shape[0]):

            row_data = []

            for c in range(sum_mat.shape[1]):

                if count_mat[r, c] == 0:

                    # กรณีที่ตำแหน่งนี้ ทุกไฟล์เป็น Error หรือช่องว่างหมดเลย ให้คืนค่าเป็นช่องว่าง

                    row_data.append("")

                else:

                    # เอาผลรวม หารด้วย จำนวนไฟล์ที่ "มีข้อมูลตัวเลข" ในช่องนี้เท่านั้น

                    avg_val = sum_mat[r, c] / count_mat[r, c]

                    row_data.append("" if avg_val == 0 else round(avg_val, 2))

            result.append(row_data)

        return result



    if file_count1 > 0:

        avg_data1 = format_average(sum_matrix1, count_matrix1)

        summary_sheet.update(range_name=TARGET_RANGE1, values=avg_data1)

        print(f"✅ ดึงและประมวลผลสำเร็จ ({file_count1} ไฟล์ต้นทาง)")

    else:

        print("❌ ไม่พบข้อมูล/ดึงไม่สำเร็จ")



    print("="*30 + "\n✅ เสร็จสิ้นกระบวนการทั้งหมด!")



# รันฟังก์ชัน

if __name__ == "__main__":

    main()



🔄 กำลังตรวจสอบสิทธิ์การเข้าถึงบัญชี Google...
✅ เข้าสู่ระบบสำเร็จ

กำลังเริ่มดึงข้อมูลของ Setup... ระบบกำลังประมวลผล


📊 ผลการดึงข้อมูล:
✅ ดึงและประมวลผลสำเร็จ (46 ไฟล์ต้นทาง)
✅ เสร็จสิ้นกระบวนการทั้งหมด!


**WD30**

In [ ]:
# 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น
import gspread
import numpy as np
import re
from google.colab import auth
from google.auth import default

# ==========================================
# ⚠️ ตั้งค่าตรงนี้ก่อนรัน (ตั้งค่า URL และ ช่วงเซลล์)
# ==========================================
# ใส่ URL ของไฟล์หลักของคุณ
MAIN_SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1vicbYhMVNJN_oRCF2DN-9LhJwufKSMkh-LzVrl7qqGc/edit?usp=sharing"

# ระบุช่วงเซลล์ "ต้นทาง" และ "ปลายทาง" สำหรับตารางที่ 1 ของ WD30
SOURCE_RANGE1 = "I3:V26"
TARGET_RANGE1 = "C5:P28"

# ตั้งชื่อชีตให้ตรงกับไฟล์หลัก (สำหรับ WD30)
SETUP_SHEET_NAME = "Setup"
SUMMARY_SHEET_NAME = "Summary_WD30"
# ==========================================

# ฟังก์ชันแปลง A1 Notation เพื่อหาขนาด (แถว, คอลัมน์) ที่แน่นอน
def get_dimensions(range_str):
    match = re.match(r'([A-Z]+)(\d+):([A-Z]+)(\d+)', range_str)
    if match:
        c1, r1, c2, r2 = match.groups()
        def col_to_num(col):
            return sum((ord(char) - 64) * (26 ** i) for i, char in enumerate(reversed(col)))
        return (int(r2) - int(r1) + 1, col_to_num(c2) - col_to_num(c1) + 1)
    return 0, 0

# ฟังก์ชันดึงค่าและตรวจสอบว่าใช้คำนวณได้หรือไม่ (แยกค่า Error ออก)
def parse_values(raw_values, expected_rows, expected_cols):
    matrix = np.zeros((expected_rows, expected_cols))
    valid_mask = np.zeros((expected_rows, expected_cols)) # เก็บค่า 1 ถ้าเป็นตัวเลขที่นำมาคำนวณได้

    for r in range(min(expected_rows, len(raw_values))):
        for c in range(min(expected_cols, len(raw_values[r]))):
            val = raw_values[r][c]

            if isinstance(val, (int, float)):
                matrix[r, c] = val
                valid_mask[r, c] = 1 # นับว่าเป็นค่าที่ถูกต้อง

            elif isinstance(val, str):
                val_str = val.strip()
                # ข้ามค่าว่างและ Error ของ Google Sheets (ที่ขึ้นต้นด้วย # เช่น #N/A, #DIV/0!, #REF!)
                if val_str == "" or val_str.startswith('#'):
                    continue

                try:
                    matrix[r, c] = float(val_str.replace(',', ''))
                    valid_mask[r, c] = 1 # นับว่าเป็นค่าที่ถูกต้อง
                except ValueError:
                    pass # หากแปลงเป็นตัวเลขไม่ได้ ให้ข้ามไป

    return matrix, valid_mask

def main():
    print("🔄 กำลังตรวจสอบสิทธิ์การเข้าถึงบัญชี Google...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("✅ เข้าสู่ระบบสำเร็จ\n")

    print(f"กำลังเริ่มดึงข้อมูลของ {SETUP_SHEET_NAME}... ระบบกำลังประมวลผล\n")

    try:
        main_ss = gc.open_by_url(MAIN_SPREADSHEET_URL)
    except Exception as e:
        print(f"❌ ข้อผิดพลาด: ไม่สามารถเปิดไฟล์หลักได้ โปรดตรวจสอบ URL\nรายละเอียด: {e}")
        return

    try:
        setup_sheet = main_ss.worksheet(SETUP_SHEET_NAME)
        summary_sheet = main_ss.worksheet(SUMMARY_SHEET_NAME)
    except gspread.exceptions.WorksheetNotFound as e:
        print(f"❌ ข้อผิดพลาด: หาชีตชื่อไม่พบ ({e})\nโปรดตรวจสอบว่าตั้งชื่อแท็บถูกต้องและไม่มีช่องว่างเกินมา")
        return

    # ดึงข้อมูลจาก Setup (ข้ามแถวแรกที่เป็น Header)
    setup_data = setup_sheet.get_all_values()[1:]

    ROWS1, COLS1 = get_dimensions(SOURCE_RANGE1)

    # เก็บผลรวม
    sum_matrix1 = np.zeros((ROWS1, COLS1))

    # เก็บจำนวนครั้งที่พบข้อมูลถูกต้อง (Cell-by-cell counting)
    count_matrix1 = np.zeros((ROWS1, COLS1))

    file_count1 = 0

    for i, row in enumerate(setup_data):
        # 📌 ดึง URL จากคอลัมน์ B (Index 1) และ ชื่อชีตจากคอลัมน์ D (Index 3)
        url1 = row[1].strip() if len(row) > 1 else ""
        sheet_name1 = row[3].strip() if len(row) > 3 else ""

        # 📌 ตรวจสอบค่า Checkbox ในคอลัมน์ E (Index 4) ว่าถูกติ๊กหรือไม่
        is_checked = False
        if len(row) > 4:
            is_checked = row[4].strip().upper() == 'TRUE'

        row_num = i + 2 # อ้างอิงตามบรรทัดใน Google Sheets

        # 📌 เงื่อนไข: ถ้ามีทั้ง URL, ชื่อชีต และ Checkbox ในคอลัมน์ E ถูกติ๊ก (TRUE) ถึงจะดึงข้อมูล
        if url1 and sheet_name1 and is_checked:
            try:
                source_ss1 = gc.open_by_url(url1)
                source_sheet1 = source_ss1.worksheet(sheet_name1)

                raw_data1 = source_sheet1.get(SOURCE_RANGE1, value_render_option='UNFORMATTED_VALUE')

                matrix1, valid1 = parse_values(raw_data1, ROWS1, COLS1)
                sum_matrix1 += matrix1
                count_matrix1 += valid1 # บวกจำนวนตัวหารเฉพาะช่องที่มีค่า
                file_count1 += 1

            except gspread.exceptions.WorksheetNotFound:
                print(f"⚠️ ไม่พบชีตชื่อ '{sheet_name1}' ในไฟล์บรรทัดที่ {row_num}")
            except Exception as e:
                print(f"⚠️ Error (บรรทัด {row_num}): {e}")

        # แจ้งเตือนกรณีที่มี URL และชื่อชีต แต่ไม่ได้ติ๊ก Checkbox
        elif url1 and sheet_name1 and not is_checked:
            print(f"⏩ ข้ามบรรทัดที่ {row_num}: เนื่องจากไม่ได้ติ๊กเลือกในคอลัมน์ E")

    # ==========================================
    # สรุปและเขียนกลับไปยัง Summary
    # ==========================================
    print("\n" + "="*30)
    print("📊 ผลการดึงข้อมูล:")

    # ฟังก์ชันหาค่าเฉลี่ย โดยพิจารณาตัวหารแบบช่องต่อช่อง
    def format_average(sum_mat, count_mat):
        result = []
        for r in range(sum_mat.shape[0]):
            row_data = []
            for c in range(sum_mat.shape[1]):
                if count_mat[r, c] == 0:
                    # กรณีที่ตำแหน่งนี้ ทุกไฟล์เป็น Error หรือช่องว่างหมดเลย ให้คืนค่าเป็นช่องว่าง
                    row_data.append("")
                else:
                    # เอาผลรวม หารด้วย จำนวนไฟล์ที่ "มีข้อมูลตัวเลข" ในช่องนี้เท่านั้น
                    avg_val = sum_mat[r, c] / count_mat[r, c]
                    row_data.append("" if avg_val == 0 else round(avg_val, 2))
            result.append(row_data)
        return result

    if file_count1 > 0:
        avg_data1 = format_average(sum_matrix1, count_matrix1)
        summary_sheet.update(range_name=TARGET_RANGE1, values=avg_data1)
        print(f"✅ ดึงและประมวลผลสำเร็จ ({file_count1} ไฟล์ต้นทาง)")
    else:
        print("❌ ไม่พบข้อมูล/ดึงไม่สำเร็จ")

    print("="*30 + "\n✅ เสร็จสิ้นกระบวนการทั้งหมด!")

# รันฟังก์ชัน
if __name__ == "__main__":
    main()

🔄 กำลังตรวจสอบสิทธิ์การเข้าถึงบัญชี Google...
✅ เข้าสู่ระบบสำเร็จ

กำลังเริ่มดึงข้อมูลของ Setup... ระบบกำลังประมวลผล


📊 ผลการดึงข้อมูล:
✅ ดึงและประมวลผลสำเร็จ (46 ไฟล์ต้นทาง)
✅ เสร็จสิ้นกระบวนการทั้งหมด!
